# JetRacer Semantic Lane Following - Live UI

YOLO26n-sem phân loại dense mask `road / divider / forbidden / obstacle`, sau đó planner chọn divider cam hoặc corridor né an toàn. Mặc định **DISARMED**; kê bánh xe và kiểm tra chiều lái trước khi bật motor. Chạy các cell từ trên xuống.

In [ ]:
import sys, time, csv, threading
from pathlib import Path
import cv2, numpy as np, ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'yolo_lane_following' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from notebook3.basic_motion import JetRacerController
from yolo_lane_following.config import load_config
from yolo_lane_following.semantic_perception import YoloSemanticPerception
from yolo_lane_following.control import AdaptiveController

cfg = load_config(PROJECT_ROOT / 'yolo_lane_following/config.yaml')
engine = PROJECT_ROOT / 'yolo_lane_following/artifacts/track_yolo26n_sem_nano_fp16.engine'
if engine.exists():
    cfg['models']['semantic'] = str(engine)
    backend_name = 'TensorRT FP16'
else:
    cfg['models']['semantic'] = str(PROJECT_ROOT / 'yolo_lane_following/artifacts/track_yolo26n_sem_best.pt')
    cfg['models']['device'] = '0'
    backend_name = 'PyTorch fallback'
perception = YoloSemanticPerception(cfg)
controller = AdaptiveController(dict(cfg['control'], max_lost_frames=cfg['tracking']['max_lost_frames']))
print('Semantic backend:', backend_name)

Nếu camera CSI bị khóa, chạy `sudo systemctl restart nvargus-daemon` trong terminal trước cell tiếp theo. Notebook không nhúng mật khẩu sudo.

In [ ]:
try:
    camera.running = False; camera.unobserve_all()
except Exception:
    pass
camera = CSICamera(width=224, height=224, capture_fps=0)
car = JetRacerController(cfg['control']['steering_gain'], cfg['control']['steering_offset'], cfg['control']['throttle_gain'], cfg['control']['throttle_max'])
car.stop(); car.center_steering()
snapshot = None


In [ ]:
state = widgets.ToggleButtons(options=['stop','live'], value='stop', description='State')
armed = widgets.Checkbox(value=False, description='ARM MOTOR')
max_throttle = widgets.FloatSlider(value=cfg['control']['throttle_max'], min=0.0, max=0.5, step=0.005, description='Max throttle')
steering_scale = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description='Steer scale')
raw_view = widgets.Image(format='jpeg', width=224, height=224)
debug_view = widgets.Image(format='jpeg', width=224, height=224)
status = widgets.HTML(value='<b>STOPPED / DISARMED</b>')
blank = np.zeros((224,224,3), np.uint8); raw_view.value=bgr8_to_jpeg(blank); debug_view.value=bgr8_to_jpeg(blank)
display(widgets.VBox([widgets.HBox([raw_view, debug_view]), status, widgets.HBox([state, armed]), max_throttle, steering_scale]))

In [ ]:
callback_lock = threading.Lock(); last_tick = time.perf_counter(); fps_ema = 0.0
log_dir = PROJECT_ROOT / 'yolo_lane_following/logs'; log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / time.strftime('semantic_live_%Y%m%d_%H%M%S.csv')
log_stream = log_path.open('w', newline=''); log_writer = csv.writer(log_stream); log_buffer=[]
log_writer.writerow(['timestamp','fps','lane_confidence','target_x','risk','steering','throttle','state','armed'])

def live_update(change):
    global last_tick, fps_ema
    if state.value != 'live' or not callback_lock.acquire(False): return
    try:
        frame = change['new']; now = time.perf_counter(); dt=max(0.005, now-last_tick); last_tick=now
        result = perception.infer(frame)
        command = controller.update(result.lane, result.obstacle_risk, frame.shape[1], dt)
        command.steering = float(np.clip(command.steering * steering_scale.value, -1, 1))
        command.throttle = min(command.throttle, max_throttle.value)
        safe_state = command.state in ('follow', 'avoid')
        if armed.value and safe_state:
            car.set_steering(command.steering); car.set_throttle(command.throttle)
        else:
            car.stop(); car.center_steering()
        instant=1.0/dt; fps_ema=instant if fps_ema==0 else .2*instant+.8*fps_ema
        rendered=result.annotated.copy(); cv2.putText(rendered, command.state, (5,16), cv2.FONT_HERSHEY_SIMPLEX, .45, (0,255,255), 1, cv2.LINE_AA)
        raw_view.value=bgr8_to_jpeg(frame); debug_view.value=bgr8_to_jpeg(rendered)
        status.value='<b>%s | %s | FPS %.1f | conf %.2f | risk %.2f | steer %+.3f | throttle %.3f</b>' % (backend_name, command.state, fps_ema, result.lane.confidence, result.obstacle_risk, command.steering, command.throttle)
        log_buffer.append([time.time(), fps_ema, result.lane.confidence, result.lane.target_x, result.obstacle_risk, command.steering, command.throttle, command.state, int(armed.value)])
        if len(log_buffer) >= 20: log_writer.writerows(log_buffer); log_stream.flush(); log_buffer.clear()
    except Exception as exc:
        car.stop(); car.center_steering(); state.value='stop'; status.value='<b style="color:red">ERROR: %s</b>' % exc
    finally:
        callback_lock.release()

def safety_changed(change):
    if change.get('new') == 'stop' or not armed.value:
        car.stop(); car.center_steering()
        if change.get('new') == 'stop' and log_buffer: log_writer.writerows(log_buffer); log_stream.flush(); log_buffer.clear()
state.observe(safety_changed, names='value'); armed.observe(safety_changed, names='value')
camera.observe(live_update, names='value'); camera.running=True
print('Camera running. Chọn live để infer; ARM MOTOR chỉ sau khi kê bánh và kiểm tra mask trắng.')

## Dừng an toàn - luôn chạy cell này trước khi đóng notebook

In [ ]:
state.value='stop'; armed.value=False; car.stop(); car.center_steering()
camera.running=False; camera.unobserve_all()
if log_buffer: log_writer.writerows(log_buffer); log_buffer.clear()
log_stream.flush(); log_stream.close()
print('Stopped safely. Log:', log_path)